In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import lightgbm as lgb
from lightgbm import early_stopping

# 全局中文设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 读取划分数据集
df = pd.read_csv("train.csv")
X = df.drop(['id','target'],axis=1)
y = df['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
cat_cols = [f'cat_{i}' for i in range(20)]
num_cols = [f'num_{i}' for i in range(38)]

In [ ]:
# ======逻辑回归======
# 1.特征预处理
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)

# 2.初始化逻辑回归
lr_model = LogisticRegression(max_iter=2000, random_state=42)

# 3.模型训练
lr_model.fit(X_train_lr, y_train)

# 4.输出5G用户预测概率
lr_y_proba = lr_model.predict_proba(X_test_lr)[:,1]

# 5.计算AUC
lr_auc = roc_auc_score(y_test, lr_y_proba)
print(f"【逻辑回归】测试集AUC = {lr_auc:.4f}")

# 绘制逻辑回归ROC
fpr_lr,tpr_lr,_ = roc_curve(y_test,lr_y_proba)
plt.figure(figsize=(6,4))
plt.plot(fpr_lr,tpr_lr,label=f'逻辑回归AUC:{lr_auc:.4f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('假阳率(FPR)')
plt.ylabel('真阳率(TPR)')
plt.title("逻辑回归ROC曲线图")
plt.legend()
plt.tight_layout() # 自动对齐画布
plt.show()

In [ ]:
# ======LightGBM======
# 1.初始化LightGBM
lgb_model = lgb.LGBMClassifier(random_state=42,learning_rate=0.05,n_estimators=200,verbose=-1)

# 2.模型训练
lgb_model.fit(X_train,y_train,
              eval_set=[(X_test,y_test)],
              callbacks=[early_stopping(stopping_rounds=30)])

# 3.输出5G用户预测概率
lgb_y_proba = lgb_model.predict_proba(X_test)[:,1]

# 4.计算AUC
lgb_auc = roc_auc_score(y_test, lgb_y_proba)
print(f"【LightGBM】测试集AUC = {lgb_auc:.4f}")

# 特征重要性图（排版对齐）
plt.figure(figsize=(10,5))
lgb.plot_importance(lgb_model,max_num_features=15,title="LightGBM前15项特征重要性")
plt.xlabel('特征增益重要度')
plt.ylabel('特征名称')
plt.tight_layout()
plt.show()

# LightGBM单独ROC
fpr_lgb,tpr_lgb,_ = roc_curve(y_test,lgb_y_proba)
plt.figure(figsize=(6,4))
plt.plot(fpr_lgb,tpr_lgb,label=f'LightGBM AUC:{lgb_auc:.4f}',c='red')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('假阳率(FPR)')
plt.ylabel('真阳率(TPR)')
plt.title("LightGBM ROC曲线图")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 汇总保存预测结果
result_df = pd.DataFrame({
    '样本序号':range(len(y_test)),
    '真实标签(1=5G用户，0=非5G用户)':y_test.values,
    '逻辑回归_5G开通概率':np.round(lr_y_proba,4),
    'LightGBM_5G开通概率':np.round(lgb_y_proba,4)
})
# 保存csv
result_df.to_csv("5G用户预测概率结果.csv",index=False,encoding='utf-8-sig')
print("预测概率已保存：5G用户预测概率结果.csv")

pd.reset_option('^display', silent=True)
pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.width', 1300)
pd.set_option('display.max_columns', None)
pd.set_option('display.colheader_justify','center')

print(result_df.head(10).to_string(index=False))